[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-09-test-driven-data-dev.ipynb#scrollTo=aa1b2c3d)

---
# Day 9 · Test-Driven Data Development — Writing Checks Before Transformations
**certified-journeys / sodacore-certified** · Review · Red-green, pre/post checks, scan variables, quality gate matrix

> **Goal for today:** Apply the red-green TDD workflow to data pipelines — write failing checks first, implement the transformation, then confirm the checks go green — and design a reusable quality gate matrix.


In [ ]:
%pip install -q soda-core-duckdb


## Step 1 · What Is Test-Driven Data Development (TDDD)?

In software TDD you write a failing test **before** writing the code that makes it pass. TDDD applies the same discipline to data pipelines:

| TDD (software) | TDDD (data) |
|---|---|
| Write a failing unit test | Write a failing Soda check for a column that doesn't exist yet |
| Implement the function | Run the transformation that creates the column |
| Test goes green | Re-run the scan — check passes |
| Refactor safely | Change transformation code; checks protect regressions |

**Benefits:**
- Forces you to define the expected output *before* writing SQL — catches design flaws early.
- Checks become living documentation of what each table must contain.
- Failed checks in CI prevent bad transformations from reaching production.

The pattern in this notebook:
1. **Red:** write a check for a column that doesn't exist — scan FAILS
2. **Green:** create the column (run the transformation) — scan PASSES
3. **Refactor:** parameterise by date partition using scan variables


In [ ]:
import duckdb, tempfile, pathlib
from soda.scan import Scan

tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "tddd.duckdb")

# ── Create raw source table (no revenue_usd yet) ────────────────────────────
conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE raw_orders (
        id          INTEGER,
        customer_id INTEGER,
        amount_eur  DOUBLE,
        created_at  DATE
    )
""")
conn.execute("""
    INSERT INTO raw_orders VALUES
        (1, 101, 45.00, '2024-01-15'),
        (2, 102, 80.00, '2024-01-15'),
        (3, 103, 10.00, '2024-01-16'),
        (4, NULL, 25.00, '2024-01-16'),
        (5, 105, 60.00, '2024-01-17')
""")
conn.close()

config_yml = f"""
data_sources:
  mydb:
    type: duckdb
    path: "{db_path}"
"""
config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)
print("Raw orders table created (no revenue_usd column yet)")


**What just happened?**

- `raw_orders` is the source table — it has `amount_eur` but **no `revenue_usd` column** yet.
- Row 4 has a `NULL` `customer_id` — a data quality issue we'll catch with a pre-transformation check.
- The transformation task (Step 3) will convert EUR to USD and populate `revenue_usd`.


## Step 2 · RED — Write the Failing Check First

Before writing any SQL, we write the checks that the transformed table **must** satisfy. Running these checks now will produce FAIL outcomes — that is the intended red state.

```yaml
# Post-transformation checks (written BEFORE the transformation exists)
checks for orders_usd:
  - row_count > 0
  - missing_count(revenue_usd) = 0
  - min(revenue_usd) > 0
  - duplicate_count(id) = 0
```

The table `orders_usd` doesn't exist yet — the scan will fail with a table-not-found error. That is our red signal.


In [ ]:
# ── RED: write post-transformation checks; table doesn't exist yet ───────────
post_checks_yml = """
checks for orders_usd:
  - row_count > 0
  - missing_count(revenue_usd) = 0
  - min(revenue_usd) > 0
  - duplicate_count(id) = 0
"""
post_checks_path = tmpdir / "post_checks.yml"
post_checks_path.write_text(post_checks_yml)

scan_red = Scan()
scan_red.set_data_source_name("mydb")
scan_red.add_configuration_yaml_file(str(config_path))
scan_red.add_sodacl_yaml_file(str(post_checks_path))
scan_red.execute()

print("=== RED STATE ===")
print("Has fails:", scan_red.has_check_fails())
print("Logs:")
print(scan_red.get_logs_text())


**What just happened?**

- The scan fails because `orders_usd` does not exist — this is the **red state**.
- `has_check_fails()` returns `True` — a CI gate would block the pipeline here.
- **This is intentional:** the checks are the specification; the transformation must satisfy them.
- Notice we wrote the checks **in YAML** before touching any SQL — the checks define the contract.


## Step 3 · Pre-Transformation Checks on Raw Source Data

Before running the transformation, validate the **source data** to fail fast on upstream issues. If the source is dirty, the transformation output will be wrong even if it runs.

These pre-transformation checks form the **input contract** of your pipeline stage:

```yaml
checks for raw_orders:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(amount_eur) = 0
  - min(amount_eur) > 0
```


In [ ]:
# ── Pre-transformation checks on raw_orders ────────────────────────────────
pre_checks_yml = """
checks for raw_orders:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(amount_eur) = 0
  - min(amount_eur) > 0
  - missing_count(customer_id):
      warn: when > 0
      fail: when > 2
"""
pre_checks_path = tmpdir / "pre_checks.yml"
pre_checks_path.write_text(pre_checks_yml)

scan_pre = Scan()
scan_pre.set_data_source_name("mydb")
scan_pre.add_configuration_yaml_file(str(config_path))
scan_pre.add_sodacl_yaml_file(str(pre_checks_path))
scan_pre.execute()

pre_outcome = "FAIL" if scan_pre.has_check_fails() else ("WARN" if scan_pre.has_check_warns() else "PASS")
print(f"Pre-transformation outcome: {pre_outcome}")
print()
print(scan_pre.get_logs_text())


**What just happened?**

- The `missing_count(customer_id)` check uses **threshold syntax**: warn if any nulls, fail if more than 2 — graded severity.
- Row 4 has `NULL` customer_id, so the scan should produce a **WARN** (1 null < threshold of 2).
- The other checks all pass — `amount_eur` is clean and `id` has no duplicates.
- **Decision point:** you can proceed to the transformation (WARN is non-blocking) but should investigate the null customer_id before marking the pipeline healthy.


## Step 4 · GREEN — Run the Transformation and Verify

Now we implement the transformation: create `orders_usd` by converting `amount_eur` to `revenue_usd` using an exchange rate, filtering out NULL customer IDs, and deduplicating by `id`.

After the transformation we re-run the same post-transformation checks from Step 2 — expecting them all to pass (green state).


In [ ]:
# ── Transformation: create orders_usd ─────────────────────────────────────
EUR_TO_USD = 1.08   # exchange rate constant

conn = duckdb.connect(db_path)
conn.execute(f"""
    CREATE TABLE orders_usd AS
    SELECT
        id,
        customer_id,
        ROUND(amount_eur * {EUR_TO_USD}, 2) AS revenue_usd,
        created_at
    FROM raw_orders
    WHERE customer_id IS NOT NULL       -- exclude records with missing customer
""")
rows = conn.execute("SELECT * FROM orders_usd ORDER BY id").fetchall()
conn.close()

print("orders_usd contents:")
print(f"  {'id':<5} {'customer_id':<15} {'revenue_usd':<15} {'created_at'}")
for row in rows:
    print(f"  {row[0]:<5} {row[1]:<15} {row[2]:<15} {row[3]}")
print()

# ── GREEN: re-run post-transformation checks ───────────────────────────────
scan_green = Scan()
scan_green.set_data_source_name("mydb")
scan_green.add_configuration_yaml_file(str(config_path))
scan_green.add_sodacl_yaml_file(str(post_checks_path))
scan_green.execute()

green_outcome = "FAIL" if scan_green.has_check_fails() else ("WARN" if scan_green.has_check_warns() else "PASS")
print(f"=== GREEN STATE ===")
print(f"Post-transformation outcome: {green_outcome}")
print(scan_green.get_logs_text())


**What just happened?**

- `orders_usd` now exists with `revenue_usd = amount_eur * 1.08`, rounded to 2 decimal places.
- The NULL `customer_id` row was filtered out — 4 rows remain instead of 5.
- **All four post-transformation checks now pass** — we have gone from red to green.
- The same check file was reused without modification — confirming the transformation satisfies the pre-written contract.


## Step 5 · Parameterise Checks with Scan Variables

In production pipelines, you often process data for a specific date partition. Soda scan variables let you inject runtime parameters into SodaCL checks using `${variable_name}` syntax.

```yaml
# partition_checks.yml
checks for orders_usd:
  - row_count > 0:
      filter: created_at = '${partition_date}'
  - missing_count(revenue_usd) = 0:
      filter: created_at = '${partition_date}'
```

Pass variables in Python with `scan.add_variables({"partition_date": "2024-01-15"})`.

This pattern is essential for **incremental pipelines** where you only validate the partition being processed, not the entire table.


In [ ]:
# ── Partitioned checks using scan variables ─────────────────────────────────
partition_checks_yml = """
filter orders_usd [daily]:
  where: created_at = DATE '${partition_date}'

checks for orders_usd [daily]:
  - row_count > 0
  - missing_count(revenue_usd) = 0
  - min(revenue_usd) > 0
"""
partition_checks_path = tmpdir / "partition_checks.yml"
partition_checks_path.write_text(partition_checks_yml)

# Run for partition 2024-01-15
scan_p = Scan()
scan_p.set_data_source_name("mydb")
scan_p.add_configuration_yaml_file(str(config_path))
scan_p.add_sodacl_yaml_file(str(partition_checks_path))
scan_p.add_variables({"partition_date": "2024-01-15"})
scan_p.execute()

p_outcome = "FAIL" if scan_p.has_check_fails() else ("WARN" if scan_p.has_check_warns() else "PASS")
print(f"Partition 2024-01-15 outcome: {p_outcome}")
print(scan_p.get_logs_text())


**What just happened?**

- The `filter orders_usd [daily]` block defines a named partition filter using the variable `${partition_date}`.
- `scan.add_variables({"partition_date": "2024-01-15"})` injects the value at scan time.
- **Checks only evaluate rows matching the partition** — ideal for daily Airflow tasks that backfill one partition at a time.
- To run for a different date, simply change the variable value — the YAML stays unchanged.


## Step 6 · Quality Gate Matrix

As the number of tables grows, managing individual check files per table becomes unwieldy. A **quality gate matrix** is a Python dict that maps each table to its checks and the action to take on failure.

```python
QUALITY_GATES = {
    "raw_orders": {
        "checks_file": "pre_checks.yml",
        "on_fail": "stop",      # halt the pipeline
        "on_warn": "continue",  # log and continue
    },
    "orders_usd": {
        "checks_file": "post_checks.yml",
        "on_fail": "stop",
        "on_warn": "continue",
    },
}
```

The orchestrator iterates through this matrix in dependency order and applies the action policy.


In [ ]:
from dataclasses import dataclass, field
from typing import Literal

# ── Quality gate matrix definition ──────────────────────────────────────────
QUALITY_GATES = {
    "raw_orders": {
        "checks_file": str(pre_checks_path),
        "on_fail": "stop",
        "on_warn": "continue",
    },
    "orders_usd": {
        "checks_file": str(post_checks_path),
        "on_fail": "stop",
        "on_warn": "continue",
    },
}


def run_quality_gate_matrix(
    gates: dict,
    config_path: pathlib.Path,
    datasource: str = "mydb",
    variables: dict = None,
) -> list:
    """Run each table's quality gate in order; return a list of gate results."""
    results = []
    for table, gate in gates.items():
        scan = Scan()
        scan.set_data_source_name(datasource)
        scan.add_configuration_yaml_file(str(config_path))
        scan.add_sodacl_yaml_file(gate["checks_file"])
        if variables:
            scan.add_variables(variables)
        scan.execute()

        has_fail = scan.has_check_fails()
        has_warn = scan.has_check_warns()
        outcome  = "FAIL" if has_fail else ("WARN" if has_warn else "PASS")
        action   = gate["on_fail"] if has_fail else (gate["on_warn"] if has_warn else "continue")

        result = {"table": table, "outcome": outcome, "action": action}
        results.append(result)

        print(f"  Gate [{table}]: {outcome} → action: {action}")

        if action == "stop" and has_fail:
            print(f"  Pipeline stopped at gate '{table}'")
            break   # halt on first FAIL+stop gate

    return results


print("Running quality gate matrix...")
gate_results = run_quality_gate_matrix(QUALITY_GATES, config_path)
print("\nGate summary:")
for r in gate_results:
    print(f"  {r['table']:<20} {r['outcome']:<6}  action={r['action']}")


**What just happened?**

- `run_quality_gate_matrix` iterates the gates dict in insertion order, running each scan in sequence.
- The matrix separates **what to check** (the YAML file) from **what to do** (the action policy) — clean separation of concerns.
- A `"stop"` action on FAIL breaks the loop — tables downstream of a failed gate are not checked (they'd inherit garbage anyway).
- **In Airflow:** each gate becomes a separate task with `on_failure_callback` wired to the action policy.


## Step 7 · TDDD Workflow Summary

Here is the complete red-green cycle as a checklist for any new pipeline stage:

1. **Define the output contract** — write `checks for <output_table>` in YAML before any SQL.
2. **Run the checks (RED)** — confirm they fail (table doesn't exist or column missing).
3. **Write pre-transformation checks** — validate the source data input contract.
4. **Implement the transformation** — write the SQL/dbt model that satisfies the output contract.
5. **Run the checks (GREEN)** — confirm all post-transformation checks pass.
6. **Parameterise with variables** — add `filter` blocks for date partitions.
7. **Register in quality gate matrix** — add the table's gate to the orchestration config.

> **Tip:** Keep pre-checks and post-checks in separate YAML files. Pre-checks run before the transformation task; post-checks run after. This lets you use them independently in an orchestrator without coupling their execution.


In [ ]:
# Challenge: Add a third table 'daily_revenue' to the pipeline and the gate matrix.
#
# 1. Create daily_revenue as an aggregation of orders_usd:
#    SELECT created_at, SUM(revenue_usd) AS total_revenue, COUNT(*) AS order_count
#    FROM orders_usd GROUP BY created_at
#
# 2. Write a checks YAML for daily_revenue:
#    - row_count > 0
#    - missing_count(total_revenue) = 0
#    - min(total_revenue) > 0
#
# 3. Add daily_revenue to QUALITY_GATES with on_fail='stop', on_warn='continue'
#
# 4. Run run_quality_gate_matrix with the updated gates dict
#
# Scaffold:

# conn = duckdb.connect(db_path)
# conn.execute("CREATE TABLE daily_revenue AS ...")
# conn.close()

# daily_checks_yml = """
# checks for daily_revenue:
#   ...
# """
# ...


---
## Day 9 key concepts recap
| Concept | What to remember |
|---|---|
| Red-green TDDD cycle | Write failing checks → implement transformation → verify checks pass |
| Pre-transformation checks | Validate source data input contract before running SQL |
| Post-transformation checks | Validate output table against the pre-written contract |
| Scan variables | `${var}` in SodaCL + `scan.add_variables({})` for partition parameterisation |
| `filter` block | Scopes checks to a named subset; combine with `${partition_date}` for daily gates |
| Quality gate matrix | Python dict: table → checks_file + on_fail/on_warn policy; iterate in dependency order |

> **Tip:** Write your post-transformation checks **in the same PR as the transformation SQL**. Reviewers can verify the contract and the implementation together — no separate documentation needed.

---
## What's next
**Day 10** → Capstone — build a full data quality framework for a multi-table e-commerce pipeline, combining everything from Days 1–9.

Mark Day 9 complete in your [tracker](../index.html).
